# ForeForest regression test (Bagging vs GBDT)

This notebook runs `foreforest` on synthetic regression data in both modes:
- `Mode.Bagging`
- `Mode.GBDT`

Run top-to-bottom.

In [1]:
from pathlib import Path
import sys
import time
import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

ROOT = Path.cwd().resolve()
cand_order = [
    ROOT / 'build',
    ROOT.parent / 'build',
    ROOT / 'tree' / 'build',
    ROOT.parent / 'tree' / 'build',
]
existing = []
for cand in cand_order:
    if cand.exists() and cand not in existing:
        existing.append(cand)
for cand in reversed(existing):
    sys.path.insert(0, str(cand))

import foreforest

print('foreforest:', foreforest.__file__)
print('numpy:', np.__version__)
print('pandas:', pd.__version__)

foreforest: /data/back_home/baseline/foreblocks/tree/build/foreforest.cpython-312-x86_64-linux-gnu.so
numpy: 2.3.5
pandas: 2.3.3


In [2]:
def mse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.mean((y_true - y_pred) ** 2))


def make_regression_data(n=24000, p=16, seed=42, missing_rate=0.03):
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(n, p)).astype(np.float64)

    X[:, 0] = rng.integers(0, 6, size=n)
    X[:, 1] = rng.integers(0, 4, size=n)

    y = (
        2.0 * (X[:, 0] == 3)
        - 1.4 * (X[:, 0] == 1)
        + 1.1 * (X[:, 1] == 2)
        + 0.9 * X[:, 2] * X[:, 3]
        - 1.2 * np.sin(X[:, 4])
        + 0.3 * (X[:, 5] ** 2)
        + rng.normal(0.0, 0.35, size=n)
    ).astype(np.float64)

    miss = rng.random((n, p)) < missing_rate
    X[miss] = np.nan
    return X, y


def split_tvt(X, y, train_ratio=0.70, valid_ratio=0.15, seed=99):
    n = X.shape[0]
    perm = np.random.default_rng(seed).permutation(n)
    n_train = int(train_ratio * n)
    n_valid = int(valid_ratio * n)
    i_train = perm[:n_train]
    i_valid = perm[n_train:n_train + n_valid]
    i_test = perm[n_train + n_valid:]
    return X[i_train], y[i_train], X[i_valid], y[i_valid], X[i_test], y[i_test]


def build_hist_cfg():
    hcfg = foreforest.HistogramConfig()
    hcfg.method = 'adaptive'
    hcfg.max_bins = 256
    hcfg.use_missing_bin = True
    hcfg.adaptive_binning = True
    hcfg.use_parallel = False
    return hcfg


def build_tree_cfg():
    cfg = foreforest.TreeConfig()
    cfg.max_depth = 8
    cfg.max_leaves = 63
    cfg.min_samples_split = 20
    cfg.min_samples_leaf = 10
    cfg.min_child_weight = 1e-3
    cfg.lambda_ = 1.0
    cfg.alpha_ = 0.0
    cfg.gamma_ = 0.0
    cfg.growth = foreforest.Growth.LeafWise
    cfg.split_mode = foreforest.SplitMode.Histogram
    cfg.exact_cutover = 2048
    cfg.enable_kway_splits = False
    cfg.enable_oblique_splits = False
    cfg.subsample_bytree = 1.0
    cfg.subsample_bylevel = 1.0
    cfg.subsample_bynode = 1.0
    cfg.subsample_with_replacement = True
    cfg.goss.enabled = True
    return cfg


def build_forest_cfg(mode):
    cfg = foreforest.ForeForestConfig()
    cfg.mode = mode
    cfg.objective = foreforest.Objective.SquaredError
    cfg.n_estimators = 300
    cfg.learning_rate = 0.05
    cfg.rng_seed = 123

    cfg.colsample_bytree = 0.8
    cfg.colsample_bynode = 0.8

    cfg.rf_row_subsample = 0.9
    cfg.rf_bootstrap = True
    cfg.rf_parallel = True

    cfg.gbdt_use_subsample = True
    cfg.gbdt_row_subsample = 0.8
    cfg.early_stopping_enabled = True
    cfg.early_stopping_rounds = 30
    cfg.early_stopping_min_delta = 0.0
    cfg.dart_enabled = False

    cfg.fw_use_subsample = True
    cfg.fw_row_subsample = 0.8
    cfg.fw_nu = 0.05
    cfg.fw_line_search_points = 12
    cfg.fw_alpha_max = 3.0
    cfg.fw_alpha_tol = 1e-8

    cfg.hist_cfg = build_hist_cfg()
    cfg.tree_cfg = build_tree_cfg()
    return cfg


def run_foreforest_mode(mode, X_train, y_train, X_valid, y_valid, X_test, y_test):
    cfg = build_forest_cfg(mode)
    model = foreforest.ForeForest(cfg)

    t0 = time.perf_counter()
    model.fit_complete(X_train, y_train, X_valid, y_valid)
    fit_s = time.perf_counter() - t0

    t1 = time.perf_counter()
    pred_valid = np.asarray(model.predict(X_valid), dtype=np.float64)
    pred_test = np.asarray(model.predict(X_test), dtype=np.float64)
    pred_s = time.perf_counter() - t1

    best_iter = int(model.best_iteration()) if hasattr(model, 'best_iteration') else -1
    early_stop = bool(model.early_stopped()) if hasattr(model, 'early_stopped') else False

    return {
        'model': str(mode),
        'fit_s': fit_s,
        'pred_s': pred_s,
        'mse_valid': mse(y_valid, pred_valid),
        'mse_test': mse(y_test, pred_test),
        'trees': int(model.size()),
        'best_iteration': best_iter,
        'early_stopped': early_stop,
    }


def run_sklearn_hgbr(X_train, y_train, X_valid, y_valid, X_test, y_test):
    model = HistGradientBoostingRegressor(
        max_iter=300,
        learning_rate=0.05,
        max_depth=8,
        max_leaf_nodes=63,
        min_samples_leaf=10,
        l2_regularization=1.0,
        early_stopping=True,
        random_state=123,
    )

    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    fit_s = time.perf_counter() - t0

    t1 = time.perf_counter()
    pred_valid = model.predict(X_valid)
    pred_test = model.predict(X_test)
    pred_s = time.perf_counter() - t1

    n_iter = int(getattr(model, 'n_iter_', -1))
    return {
        'model': 'sklearn.HistGradientBoostingRegressor',
        'fit_s': fit_s,
        'pred_s': pred_s,
        'mse_valid': mse(y_valid, pred_valid),
        'mse_test': mse(y_test, pred_test),
        'trees': n_iter,
        'best_iteration': n_iter,
        'early_stopped': bool(n_iter < 300),
    }


def run_sklearn_random_forest(X_train, y_train, X_valid, y_valid, X_test, y_test):
    model = make_pipeline(
        SimpleImputer(strategy='median'),
        RandomForestRegressor(
            n_estimators=300,
            max_depth=8,
            min_samples_leaf=10,
            max_features=0.8,
            bootstrap=True,
            n_jobs=-1,
            random_state=123,
        ),
    )

    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    fit_s = time.perf_counter() - t0

    t1 = time.perf_counter()
    pred_valid = model.predict(X_valid)
    pred_test = model.predict(X_test)
    pred_s = time.perf_counter() - t1

    rf = model.named_steps['randomforestregressor']
    return {
        'model': 'sklearn.RandomForestRegressor',
        'fit_s': fit_s,
        'pred_s': pred_s,
        'mse_valid': mse(y_valid, pred_valid),
        'mse_test': mse(y_test, pred_test),
        'trees': int(rf.n_estimators),
        'best_iteration': -1,
        'early_stopped': False,
    }

In [3]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score


class LPBoostInspiredRegressor:
    def __init__(
        self,
        n_estimators=120,
        max_depth=3,
        learning_rate=0.1,
        subsample=0.8,
        nu=0.05,
        line_search_points=12,
        random_state=42,
        verbose=False,
    ):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.learning_rate = learning_rate
        self.subsample = subsample
        self.nu = nu
        self.line_search_points = line_search_points
        self.random_state = random_state
        self.verbose = verbose

        self.models_ = []
        self.alphas_ = []
        self.train_mse_history_ = []
        self.train_r2_history_ = []

    def fit(self, X, y):
        rng = np.random.RandomState(self.random_state)
        n_samples = len(y)

        F = np.zeros(n_samples, dtype=float)
        self.models_ = []
        self.alphas_ = []
        self.train_mse_history_ = []
        self.train_r2_history_ = []

        for m in range(self.n_estimators):
            resid = y - F

            if self.subsample < 1.0:
                sample_size = max(1, int(self.subsample * n_samples))
                idx = rng.choice(n_samples, size=sample_size, replace=False)
            else:
                idx = np.arange(n_samples)

            tree = DecisionTreeRegressor(
                max_depth=self.max_depth,
                random_state=rng.randint(1_000_000_000),
            )
            tree.fit(X[idx], resid[idx])

            h = tree.predict(X)

            best_alpha = 0.0
            best_obj = np.inf
            alphas_cand = np.linspace(0.0, 3.0, self.line_search_points)

            for alpha in alphas_cand:
                F_temp = F + alpha * h
                mse_val = mean_squared_error(y, F_temp)
                l1_penalty = self.nu * abs(alpha)
                obj = mse_val + l1_penalty
                if obj < best_obj:
                    best_obj = obj
                    best_alpha = alpha

            if best_alpha < 1e-8:
                if self.verbose:
                    print(f'Early stopping at iteration {m + 1} (alpha ≈ 0)')
                break

            effective_alpha = best_alpha * self.learning_rate
            F += effective_alpha * h

            self.models_.append(tree)
            self.alphas_.append(effective_alpha)

            current_mse = mean_squared_error(y, F)
            current_r2 = r2_score(y, F)
            self.train_mse_history_.append(current_mse)
            self.train_r2_history_.append(current_r2)

            if self.verbose and (m + 1) % 20 == 0:
                print(
                    f'Iter {m + 1:3d} | MSE: {current_mse:.5f} | R²: {current_r2:.5f} '
                    f'| |α| mean: {np.mean(np.abs(self.alphas_)):.4f} '
                    f'| trees: {len(self.models_)}'
                )

        if self.verbose and self.train_mse_history_:
            print(f'Final training MSE: {self.train_mse_history_[-1]:.5f}')
            print(f'Final training R²:  {self.train_r2_history_[-1]:.5f}')
            print(f'Number of trees:    {len(self.models_)}')
            print(f'Mean |alpha|:       {np.mean(np.abs(self.alphas_)):.4f}')

        return self

    def predict(self, X):
        pred = np.zeros(len(X), dtype=float)
        for tree, alpha in zip(self.models_, self.alphas_):
            pred += alpha * tree.predict(X)
        return pred

    def staged_predict(self, X):
        pred = np.zeros(len(X), dtype=float)
        for tree, alpha in zip(self.models_, self.alphas_):
            pred += alpha * tree.predict(X)
            yield pred.copy()


def run_lpboost_regression(X_train, y_train, X_valid, y_valid, X_test, y_test):
    model = LPBoostInspiredRegressor(
        n_estimators=120,
        max_depth=3,
        learning_rate=0.1,
        subsample=0.8,
        nu=0.05,
        line_search_points=12,
        random_state=42,
        verbose=False,
    )

    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    fit_s = time.perf_counter() - t0

    t1 = time.perf_counter()
    pred_valid = model.predict(X_valid)
    pred_test = model.predict(X_test)
    pred_s = time.perf_counter() - t1

    return {
        'model': 'LPBoostInspiredRegressor',
        'fit_s': fit_s,
        'pred_s': pred_s,
        'mse_valid': mse(y_valid, pred_valid),
        'mse_test': mse(y_test, pred_test),
        'trees': int(len(model.models_)),
        'best_iteration': int(len(model.models_)),
        'early_stopped': bool(len(model.models_) < model.n_estimators),
    }

In [4]:
X, y = make_regression_data()
X_train, y_train, X_valid, y_valid, X_test, y_test = split_tvt(X, y)

print('train:', X_train.shape, 'valid:', X_valid.shape, 'test:', X_test.shape)

train: (16800, 16) valid: (3600, 16) test: (3600, 16)


In [ ]:
results = []
# results.append(run_foreforest_mode(foreforest.Mode.Bagging, X_train, y_train, X_valid, y_valid, X_test, y_test))
results.append(run_foreforest_mode(foreforest.Mode.GBDT, X_train, y_train, X_valid, y_valid, X_test, y_test))
# results.append(run_foreforest_mode(foreforest.Mode.FWBoost, X_train, y_train, X_valid, y_valid, X_test, y_test))
results.append(run_sklearn_hgbr(X_train, y_train, X_valid, y_valid, X_test, y_test))
results.append(run_sklearn_random_forest(X_train, y_train, X_valid, y_valid, X_test, y_test))

df = pd.DataFrame(results)
df = df.sort_values(by='mse_test', ascending=True).reset_index(drop=True)
df

,model,fit_s,pred_s,mse_valid,mse_test,trees,best_iteration,early_stopped
0,Mode.GBDT,1.924951,0.061941,0.334909,0.332912,297,297,False
1,sklearn.HistGradientBoostingRegressor,0.980518,0.012859,0.377051,0.371041,300,300,False
2,sklearn.RandomForestRegressor,1.004378,0.100065,1.094325,1.049239,300,-1,False


: 

In [6]:
def run_foreforest_custom(label, X_train, y_train, X_valid, y_valid, X_test, y_test, tree_overrides=None, forest_overrides=None):
    tree_overrides = tree_overrides or {}
    forest_overrides = forest_overrides or {}

    cfg = build_forest_cfg(foreforest.Mode.Bagging)
    for key, value in tree_overrides.items():
        setattr(cfg.tree_cfg, key, value)
    for key, value in forest_overrides.items():
        setattr(cfg, key, value)

    model = foreforest.ForeForest(cfg)

    t0 = time.perf_counter()
    model.fit_complete(X_train, y_train, X_valid, y_valid)
    fit_s = time.perf_counter() - t0

    t1 = time.perf_counter()
    pred_valid = np.asarray(model.predict(X_valid), dtype=np.float64)
    pred_test = np.asarray(model.predict(X_test), dtype=np.float64)
    pred_s = time.perf_counter() - t1

    return {
        'model': label,
        'fit_s': fit_s,
        'pred_s': pred_s,
        'mse_valid': mse(y_valid, pred_valid),
        'mse_test': mse(y_test, pred_test),
        'trees': int(model.size()),
        'best_iteration': int(model.best_iteration()) if hasattr(model, 'best_iteration') else -1,
        'early_stopped': bool(model.early_stopped()) if hasattr(model, 'early_stopped') else False,
    }


diag_results = []

diag_results.append(run_foreforest_custom(
    'Bagging(base)', X_train, y_train, X_valid, y_valid, X_test, y_test
))

diag_results.append(run_foreforest_custom(
    'Bagging(lambda_=0)', X_train, y_train, X_valid, y_valid, X_test, y_test,
    tree_overrides={'lambda_': 0.0}
))

diag_results.append(run_foreforest_custom(
    'Bagging(rf_row_subsample=1.0)', X_train, y_train, X_valid, y_valid, X_test, y_test,
    forest_overrides={'rf_row_subsample': 1.0, 'rf_bootstrap': True}
))

diag_results.append(run_foreforest_custom(
    'Bagging(no_bootstrap)', X_train, y_train, X_valid, y_valid, X_test, y_test,
    forest_overrides={'rf_row_subsample': 1.0, 'rf_bootstrap': False}
))

diag_results.append(run_foreforest_custom(
    'Bagging(deeper)', X_train, y_train, X_valid, y_valid, X_test, y_test,
    tree_overrides={'max_depth': 12, 'max_leaves': 255, 'min_samples_leaf': 2}
))

diag_results.append(run_foreforest_custom(
    'Bagging(lambda0+deeper)', X_train, y_train, X_valid, y_valid, X_test, y_test,
    tree_overrides={'lambda_': 0.0, 'max_depth': 12, 'max_leaves': 255, 'min_samples_leaf': 2}
))

pd.DataFrame(diag_results).sort_values('mse_test').reset_index(drop=True)

ValueError: ForeForest: GOSS currently supports GBDT mode only

In [ ]:
results_tuned = []
results_tuned.append(run_foreforest_mode(foreforest.Mode.GBDT, X_train, y_train, X_valid, y_valid, X_test, y_test))
results_tuned.append(run_sklearn_hgbr(X_train, y_train, X_valid, y_valid, X_test, y_test))
results_tuned.append(run_sklearn_random_forest(X_train, y_train, X_valid, y_valid, X_test, y_test))
results_tuned.append(run_foreforest_custom(
    'Mode.Bagging(tuned)',
    X_train, y_train, X_valid, y_valid, X_test, y_test,
    tree_overrides={'lambda_': 0.0, 'max_depth': 12, 'max_leaves': 255, 'min_samples_leaf': 2},
))

pd.DataFrame(results_tuned).sort_values('mse_test').reset_index(drop=True)

,model,fit_s,pred_s,mse_valid,mse_test,trees,best_iteration,early_stopped
0,Mode.GBDT,1.668004,0.053546,0.338689,0.336464,300,300,False
1,sklearn.HistGradientBoostingRegressor,0.987559,0.013088,0.377051,0.371041,300,300,False
2,Mode.Bagging(tuned),0.547721,0.095877,1.041492,1.019875,300,300,False
3,sklearn.RandomForestRegressor,1.018928,0.069113,1.094325,1.049239,300,-1,False


In [ ]:
# Ablation sweep for split-gain sensitivity: lambda_, gamma_, min_child_weight
from itertools import product

lambda_grid = [0.0, 0.1, 1.0, 5.0]
gamma_grid = [0.0, 0.05, 0.2]
min_child_weight_grid = [1e-6, 1e-3, 1e-2, 1e-1]

ablation_rows = []
for lam, gam, mcw in product(lambda_grid, gamma_grid, min_child_weight_grid):
    row = run_foreforest_custom(
        label='Mode.Bagging(ablation)',
        X_train=X_train,
        y_train=y_train,
        X_valid=X_valid,
        y_valid=y_valid,
        X_test=X_test,
        y_test=y_test,
        tree_overrides={
            'lambda_': float(lam),
            'gamma_': float(gam),
            'min_child_weight': float(mcw),
        },
    )
    row['lambda_'] = float(lam)
    row['gamma_'] = float(gam)
    row['min_child_weight'] = float(mcw)
    ablation_rows.append(row)

ablation_df = pd.DataFrame(ablation_rows).sort_values('mse_test').reset_index(drop=True)

print('Top 10 configs by mse_test:')
ablation_df[['mse_test', 'mse_valid', 'fit_s', 'lambda_', 'gamma_', 'min_child_weight']].head(10)

Top 10 configs by mse_test:


,mse_test,mse_valid,fit_s,lambda_,gamma_,min_child_weight
0,1.15568,1.193075,0.200275,0.1,0.00,0.000001
1,1.15568,1.193075,0.197202,0.1,0.00,0.001000
2,1.15568,1.193075,0.199050,0.1,0.00,0.010000
3,1.15568,1.193075,0.201212,0.1,0.00,0.100000
4,1.15568,1.193075,0.198888,0.1,0.05,0.100000
5,1.15568,1.193075,0.199904,0.1,0.05,0.010000
6,1.15568,1.193075,0.197466,0.1,0.05,0.001000
7,1.15568,1.193075,0.203436,0.1,0.05,0.000001
8,1.15568,1.193075,0.198298,0.1,0.20,0.100000
9,1.15568,1.193075,0.199600,0.1,0.20,0.010000


In [ ]:
# # Stage-2 interaction sweep: split-gain knobs + capacity/sampling knobs
# from itertools import product

# lambda_grid2 = [0.0, 0.1, 1.0]
# gamma_grid2 = [0.0, 0.1]
# min_child_weight_grid2 = [1e-6, 1e-3]
# max_depth_grid2 = [8, 12]
# max_leaves_grid2 = [63, 255]
# min_samples_leaf_grid2 = [2, 10]
# rf_row_subsample_grid2 = [0.8, 1.0]
# colsample_bytree_grid2 = [0.8, 1.0]

# stage2_rows = []
# for lam, gam, mcw, md, ml, msl, rsub, csub in product(
#     lambda_grid2,
#     gamma_grid2,
#     min_child_weight_grid2,
#     max_depth_grid2,
#     max_leaves_grid2,
#     min_samples_leaf_grid2,
#     rf_row_subsample_grid2,
#     colsample_bytree_grid2,
# ):
#     row = run_foreforest_custom(
#         label='Mode.Bagging(stage2)',
#         X_train=X_train,
#         y_train=y_train,
#         X_valid=X_valid,
#         y_valid=y_valid,
#         X_test=X_test,
#         y_test=y_test,
#         tree_overrides={
#             'lambda_': float(lam),
#             'gamma_': float(gam),
#             'min_child_weight': float(mcw),
#             'max_depth': int(md),
#             'max_leaves': int(ml),
#             'min_samples_leaf': int(msl),
#         },
#         forest_overrides={
#             'rf_row_subsample': float(rsub),
#             'colsample_bytree': float(csub),
#         },
#     )
#     row['lambda_'] = float(lam)
#     row['gamma_'] = float(gam)
#     row['min_child_weight'] = float(mcw)
#     row['max_depth'] = int(md)
#     row['max_leaves'] = int(ml)
#     row['min_samples_leaf'] = int(msl)
#     row['rf_row_subsample'] = float(rsub)
#     row['colsample_bytree'] = float(csub)
#     stage2_rows.append(row)

# stage2_df = pd.DataFrame(stage2_rows).sort_values('mse_test').reset_index(drop=True)

# print('Stage-2 best 15 configs by mse_test:')
# display(stage2_df[[
#     'mse_test', 'mse_valid', 'fit_s',
#     'lambda_', 'gamma_', 'min_child_weight',
#     'max_depth', 'max_leaves', 'min_samples_leaf',
#     'rf_row_subsample', 'colsample_bytree',
# ]].head(15))

# print('mse_test spread:')
# display(stage2_df['mse_test'].describe())

In [ ]:
# Split-finder correctness audit (SOTA-style sanity checks) in a fresh subprocess
import json
import subprocess
import textwrap
import sys

audit_code = textwrap.dedent(r'''
from pathlib import Path
import json
import numpy as np
import sys

ROOT = Path.cwd().resolve()
cand_order = [
    ROOT / 'build',
    ROOT.parent / 'build',
    ROOT / 'tree' / 'build',
    ROOT.parent / 'tree' / 'build',
]
for cand in reversed([c for c in cand_order if c.exists()]):
    sys.path.insert(0, str(cand))

import foretree

def make_splitfinder_data(n=5000, p=10, seed=1234):
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(n, p)).astype(np.float64)
    y = np.where(X[:, 0] > 0.15, 2.0, -1.5) + 0.2 * X[:, 1] * X[:, 2] + 0.05 * rng.normal(size=n)
    return X, y.astype(np.float64)

def brute_best_stump_gain(X, y, lambda_=0.1, gamma_=0.0, min_samples_leaf=10, min_child_weight=1e-3):
    n, p = X.shape
    g = -y
    h = np.ones(n, dtype=np.float64)

    def leaf_obj(G, H):
        denom = H + lambda_
        if denom <= 0.0:
            return 0.0
        return 0.5 * (G * G) / denom

    Gp = float(np.sum(g))
    Hp = float(np.sum(h))
    parent = leaf_obj(Gp, Hp)

    best = {'gain': -float('inf'), 'feature': -1, 'thr': float('nan')}
    for j in range(p):
        order = np.argsort(X[:, j])
        xs = X[order, j]
        gs = g[order]
        hs = h[order]
        pG = np.cumsum(gs)
        pH = np.cumsum(hs)

        for k in range(min_samples_leaf, n - min_samples_leaf):
            if xs[k] <= xs[k - 1]:
                continue
            GL = float(pG[k - 1])
            HL = float(pH[k - 1])
            GR = Gp - GL
            HR = Hp - HL
            if HL < min_child_weight or HR < min_child_weight:
                continue
            gain = leaf_obj(GL, HL) + leaf_obj(GR, HR) - parent - gamma_
            if gain > best['gain']:
                best = {'gain': gain, 'feature': int(j), 'thr': float(0.5 * (xs[k - 1] + xs[k]))}
    return best

def fit_single_tree(X, y, split_mode, enable_kway=False, enable_oblique=False, oblique_mode=None):
    g = (-y).astype(np.float64)
    h = np.ones_like(g)

    hcfg = foretree.HistogramConfig()
    hcfg.method = 'adaptive'
    hcfg.max_bins = 256
    hcfg.use_missing_bin = True

    ghs = foretree.GradientHistogramSystem(hcfg)
    ghs.fit_bins(X, g, h)
    Xb, _ = ghs.prebin_dataset(X)

    cfg = foretree.TreeConfig()
    cfg.max_depth = 1
    cfg.max_leaves = 2
    cfg.min_samples_split = 20
    cfg.min_samples_leaf = 10
    cfg.min_child_weight = 1e-3
    cfg.lambda_ = 0.1
    cfg.gamma_ = 0.0
    cfg.split_mode = split_mode
    cfg.enable_kway_splits = enable_kway
    cfg.enable_oblique_splits = enable_oblique
    if oblique_mode is not None:
        cfg.oblique_mode = oblique_mode

    tree = foretree.UnifiedTree(cfg, ghs)
    if split_mode != foretree.SplitMode.Histogram:
        tree.set_raw_matrix(X, None)
    tree.fit_binned(Xb, g, h)

    pred = np.asarray(tree.predict_binned(Xb, X), dtype=np.float64)
    fi = np.asarray(tree.feature_importance_gain(), dtype=np.float64)
    return {
        'mse_train': float(np.mean((pred - y) ** 2)),
        'gain_sum': float(np.sum(fi)),
        'best_feature': int(np.argmax(fi)) if fi.size else -1,
        'n_nodes': int(tree.n_nodes),
        'n_leaves': int(tree.n_leaves),
        'depth': int(tree.depth),
        'pred': pred,
    }

X_sf, y_sf = make_splitfinder_data()
brute = brute_best_stump_gain(X_sf, y_sf, lambda_=0.1, gamma_=0.0, min_samples_leaf=10, min_child_weight=1e-3)

runs = []
for name, smode, kway, oblique, omode in [
    ('hist_axis', foretree.SplitMode.Histogram, False, False, None),
    ('exact_axis', foretree.SplitMode.Exact, False, False, None),
    ('hybrid_axis', foretree.SplitMode.Hybrid, False, False, None),
    ('hist_kway', foretree.SplitMode.Histogram, True, False, None),
    ('hist_oblique_auto', foretree.SplitMode.Histogram, False, True, foretree.ObliqueMode.Auto),
]:
    out1 = fit_single_tree(X_sf, y_sf, smode, kway, oblique, omode)
    out2 = fit_single_tree(X_sf, y_sf, smode, kway, oblique, omode)
    max_abs_diff_repeat = float(np.max(np.abs(out1['pred'] - out2['pred'])))
    runs.append({
        'engine': name,
        'mse_train': out1['mse_train'],
        'gain_sum': out1['gain_sum'],
        'best_feature': out1['best_feature'],
        'depth': out1['depth'],
        'n_leaves': out1['n_leaves'],
        'deterministic_max_abs_diff': max_abs_diff_repeat,
    })

exact_row = [r for r in runs if r['engine'] == 'exact_axis'][0]
payload = {
    'brute': brute,
    'runs': runs,
    'exact_gain_minus_brute': float(exact_row['gain_sum'] - brute['gain']),
    'exact_feature_matches_brute': bool(int(exact_row['best_feature']) == int(brute['feature'])),
}
print(json.dumps(payload))
''')

out = subprocess.check_output([sys.executable, '-c', audit_code], text=True)
audit_payload = json.loads(out)

print('Brute-force best stump:', audit_payload['brute'])
print('Exact gain - brute gain:', audit_payload['exact_gain_minus_brute'])
print('Exact feature matches brute:', audit_payload['exact_feature_matches_brute'])
pd.DataFrame(audit_payload['runs']).sort_values('mse_train').reset_index(drop=True)

Brute-force best stump: {'gain': 7511.896270623564, 'feature': 0, 'thr': 0.14997702621319112}
Exact gain - brute gain: 0.0
Exact feature matches brute: True


,engine,mse_train,gain_sum,best_feature,depth,n_leaves,deterministic_max_abs_diff
0,exact_axis,0.042079,7511.896271,0,1,2,0.0
1,hist_axis,0.053260,7483.944445,0,1,2,0.0
2,hybrid_axis,0.053260,7483.944445,0,1,2,0.0
3,hist_kway,0.053260,7483.944445,0,1,2,0.0
4,hist_oblique_auto,0.053260,7483.944445,0,1,2,0.0


In [ ]:
# Binning strategy audit: dense vs sparse/high-card regression
def make_sparse_highcard_regression(n=16000, p=64, seed=777):
    rng = np.random.default_rng(seed)
    X = rng.normal(0.0, 1.0, size=(n, p)).astype(np.float64)

    # Make most features sparse
    sparse_mask = rng.random((n, p)) < 0.95
    X[sparse_mask] = 0.0

    # High-cardinality-like integer features (still numeric)
    X[:, 0] = rng.integers(0, 2000, size=n).astype(np.float64)
    X[:, 1] = rng.integers(0, 1500, size=n).astype(np.float64)

    # Target mixes sparse linear terms + high-card effects
    y = (
        0.015 * X[:, 0]
        - 0.010 * X[:, 1]
        + 2.5 * (X[:, 2] != 0.0).astype(np.float64)
        - 1.7 * (X[:, 3] != 0.0).astype(np.float64)
        + 1.2 * X[:, 4] * X[:, 5]
        + rng.normal(0.0, 0.35, size=n)
    ).astype(np.float64)
    return X, y


def run_binning_method(method_name, X_train, y_train, X_valid, y_valid, X_test, y_test):
    cfg = build_forest_cfg(foreforest.Mode.GBDT)
    cfg.n_estimators = 180
    cfg.learning_rate = 0.05
    cfg.gbdt_use_subsample = True
    cfg.gbdt_row_subsample = 0.8
    cfg.colsample_bytree = 0.8
    cfg.colsample_bynode = 0.8
    cfg.early_stopping_enabled = True
    cfg.early_stopping_rounds = 20
    cfg.early_stopping_min_delta = 0.0
    cfg.dart_enabled = False

    cfg.hist_cfg.method = method_name
    cfg.hist_cfg.max_bins = 256
    cfg.hist_cfg.min_bins = 8
    cfg.hist_cfg.target_bins = 32
    cfg.hist_cfg.adaptive_binning = True

    model = foreforest.ForeForest(cfg)

    t0 = time.perf_counter()
    model.fit_complete(X_train, y_train, X_valid, y_valid)
    fit_s = time.perf_counter() - t0

    t1 = time.perf_counter()
    pred_valid = np.asarray(model.predict(X_valid), dtype=np.float64)
    pred_test = np.asarray(model.predict(X_test), dtype=np.float64)
    pred_s = time.perf_counter() - t1

    return {
        'method': method_name,
        'fit_s': fit_s,
        'pred_s': pred_s,
        'mse_valid': mse(y_valid, pred_valid),
        'mse_test': mse(y_test, pred_test),
        'best_iteration': int(model.best_iteration()) if hasattr(model, 'best_iteration') else -1,
        'trees': int(model.size()),
    }


methods = ['hist', 'quantile', 'kmeans', 'two_stage', 'adaptive', 'grad_aware']

# Dense dataset benchmark
X_d, y_d = make_regression_data(n=18000, p=16, seed=42, missing_rate=0.03)
Xd_tr, yd_tr, Xd_va, yd_va, Xd_te, yd_te = split_tvt(X_d, y_d, train_ratio=0.7, valid_ratio=0.15, seed=99)

# Sparse/high-card benchmark
X_s, y_s = make_sparse_highcard_regression(n=18000, p=64, seed=777)
Xs_tr, ys_tr, Xs_va, ys_va, Xs_te, ys_te = split_tvt(X_s, y_s, train_ratio=0.7, valid_ratio=0.15, seed=99)

binning_rows = []
for m in methods:
    rd = run_binning_method(m, Xd_tr, yd_tr, Xd_va, yd_va, Xd_te, yd_te)
    rd['dataset'] = 'dense'
    binning_rows.append(rd)

    rs = run_binning_method(m, Xs_tr, ys_tr, Xs_va, ys_va, Xs_te, ys_te)
    rs['dataset'] = 'sparse_highcard'
    binning_rows.append(rs)

binning_df = pd.DataFrame(binning_rows).sort_values(['dataset', 'mse_test']).reset_index(drop=True)

print('Binning audit results (sorted by dataset, mse_test):')
display(binning_df[['dataset', 'method', 'mse_test', 'mse_valid', 'fit_s', 'pred_s', 'best_iteration', 'trees']])

print('Best method per dataset:')
display(
    binning_df.sort_values('mse_test').groupby('dataset', as_index=False).first()[
        ['dataset', 'method', 'mse_test', 'fit_s', 'best_iteration']
    ]
)

Binning audit results (sorted by dataset, mse_test):


,dataset,method,mse_test,mse_valid,fit_s,pred_s,best_iteration,trees
0,dense,adaptive,0.450197,0.464969,0.858350,0.027569,180,180
1,dense,kmeans,0.452149,0.457554,2.493247,0.028113,179,179
2,dense,quantile,0.454091,0.458269,1.044920,0.027807,180,180
3,dense,grad_aware,0.471480,0.468065,0.678196,0.025690,178,178
4,dense,hist,0.477565,0.480951,0.963872,0.024980,178,178
5,dense,two_stage,0.478679,0.487495,0.639283,0.024634,179,179
6,sparse_highcard,hist,0.176499,0.187002,4.427197,0.023814,180,180
7,sparse_highcard,kmeans,0.176758,0.184350,19.108430,0.025619,180,180
8,sparse_highcard,quantile,0.177461,0.181645,3.771260,0.026040,180,180
9,sparse_highcard,adaptive,0.177658,0.180201,3.811609,0.025429,180,180


Best method per dataset:


,dataset,method,mse_test,fit_s,best_iteration
0,dense,adaptive,0.450197,0.858350,180
1,sparse_highcard,hist,0.176499,4.427197,180
